In [106]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
try:                              #코랩 사용할때만 사용
  # Colab only
  pass # %tensorflow_version 2.x         #현재 코랩이 사용하는 버전이 2.x
except Exception:
  pass


Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


In [172]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

path = 'https://raw.githubusercontent.com/sehakflower/data/main/titanic.csv'
df = pd.read_csv(path, sep = '\t')

df['Age'] = df['Age'].fillna(age_mean)
print(f"- Age 평균값 : ({age_mean:.2f})으로 채움")

df['Embarked'] = df['Embarked'].fillna(embarked_mode)
print(f"- Embarked: 최빈값({embarked_mode})으로 채움")

df.drop('Cabin', axis=1, inplace=True)
print("- Cabin: 컬럼 제거 (결측치 80% 이상)")

df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.')
title_mapping = {'Mr': 1, 'Mrs': 2, 'Miss': 3, 'Master': 4}
df['Title_Encoded'] = df['Title'].map(title_mapping).fillna(5).astype(int)

df['LastName'] = df['Name'].str.split(',').str[0]
df['LastName_Group'] = df['LastName'].apply(
    lambda x: sum(1 for _ in range(5) if x[0].upper() > 'EJOTX'[_]) + 1)

df['Name_Length_Group'] = pd.cut(df['Name'].str.len(), bins=5, labels=[1,2,3,4,5], duplicates='drop').astype(int)

for col in ['Name', 'Title', 'LastName']:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

print("\nSex 컬럼 인코딩 완료!")


df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Fare_Person'] = df['Fare'] / df['Family_Size']

print("\n✓ Embarked, Fare_Person 처리 완료!")
print(f"\n{df[['Embarked', 'SibSp', 'Parch', 'Fare', 'Fare_Person']].head(10)}")
print(f"\n데이터 크기: {df.shape}")

df['Ticket'] = pd.factorize(df['Ticket'])[0]

df.info()


- Age 평균값 : (28.14)으로 채움
- Embarked: 최빈값(S)으로 채움
- Cabin: 컬럼 제거 (결측치 80% 이상)

Sex 컬럼 인코딩 완료!

✓ Embarked, Fare_Person 처리 완료!

   Embarked  SibSp  Parch     Fare  Fare_Person
0         0      1      0   7.2500      3.62500
1         1      1      0  71.2833     35.64165
2         0      0      0   7.9250      7.92500
3         0      1      0  53.1000     26.55000
4         0      0      0   8.0500      8.05000
5         2      0      0   8.4583      8.45830
6         0      0      0  51.8625     51.86250
7         0      3      1  21.0750      4.21500
8         0      0      2  11.1333      3.71110
9         1      1      0  30.0708     15.03540

데이터 크기: (156, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PassengerId        156 non-null    int64  
 1   Survived           156 non-null    int64  
 2   Pclass             156 no

In [146]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
import time
from tqdm import tqdm

X = df.drop(['Survived', 'PassengerId'], axis = 1)

y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt_model = DecisionTreeClassifier(random_state=42)

print("모델이 학습을 시작합니다...")


모델이 학습을 시작합니다...


In [183]:
for epoch in tqdm(range(50), desc = "[Epoch 학습 중]"):
  dt_model.fit(X_train, y_train)
  time.sleep(0.05)

print("학습이 완료되었습니다.")


[Epoch 학습 중]: 100%|██████████| 50/50 [00:03<00:00, 16.35it/s]

학습이 완료되었습니다.


In [174]:
dt_pred = dt_model.predict(X_test)
print(f"결정 트리 정확도 : {accuracy_score(y_test, dt_pred):.2f}")

결정 트리 정확도 : 0.62


In [163]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
df_model = RandomForestClassifier(random_state=42)
print("모델 학습을 시작합니다...")
for epoch in tqdm(range(10), desc = "[Epoch 학습 중]"):
    rf_model.fit(X_train, y_train)
    time.sleep(0.1)
print("모델 학습이 종료되었습니다.")


모델 학습을 시작합니다...


[Epoch 학습 중]: 100%|██████████| 10/10 [00:05<00:00,  1.93it/s]

모델 학습이 종료되었습니다.


In [164]:
rf_pred = rf_model.predict(X_test)
print(f"랜덤 포레스트 정확도 : {accuracy_score(y_test, rf_pred):.2f}")

랜덤 포레스트 정확도 : 0.72
